# Project Test Console — responsive Colab UI

Interactive controls let you add/remove any number of setup commands and tests. Configure repository, branch, Git secret and runtime; then press **Run validation**. The output shows status cards, failure details, a copyable full report and a TXT download. No Drive settings are used or saved.

**Note:** commands run in the Colab shell. Only run trusted repositories and commands. Portable Linux mock tests do not validate native Windows/MSVC hook behavior.


In [ ]:
# Project Test Console — interactive, responsive UI
import os, re, json, time, shutil, tempfile, subprocess, pathlib, platform, datetime, html, urllib.parse, base64
import ipywidgets as W
from IPython.display import display, HTML, Javascript

try:
    from google.colab import userdata
except Exception:
    userdata = None

# --- Defaults ---
repo_default = "https://github.com/Jue-dev/Tessera.git"
ref_default = "issue-6/unwind-lifecycle"
secret_default = "GH_TOKEN"

style = W.HTML("""
<style>
.ptc-root {font-family:system-ui,-apple-system,sans-serif;max-width:100%;color:var(--colab-text-color,#222)}
.ptc-card {border:1px solid #d0d7de;border-radius:12px;padding:12px;margin:8px 0}
.ptc-title {font-size:18px;font-weight:700;margin-bottom:8px}
.ptc-grid {display:flex;flex-wrap:wrap;gap:8px}
.ptc-row {display:flex;flex-wrap:wrap;gap:8px;align-items:center;border:1px solid #d0d7de;border-radius:10px;padding:8px;margin:6px 0}
.ptc-status {font-weight:700;font-size:18px}
.ptc-report {font-family:ui-monospace,monospace}
@media(max-width:600px){.ptc-row{display:block}.ptc-row>*{width:100%!important;margin:3px 0}.ptc-card{padding:9px}}
</style>
""")

repo = W.Text(value=repo_default, description="Repository", layout=W.Layout(width="100%"))
ref = W.Text(value=ref_default, description="Branch / ref", layout=W.Layout(width="100%"))
secret_name = W.Text(value=secret_default, description="Colab Secret", layout=W.Layout(width="100%"))
username = W.Text(value="x-access-token", description="Git username", layout=W.Layout(width="100%"))
depth = W.IntText(value=1, description="Clone depth", layout=W.Layout(width="160px"))
clone_timeout = W.IntText(value=240, description="Clone timeout", layout=W.Layout(width="180px"))
test_timeout = W.IntText(value=900, description="Test timeout", layout=W.Layout(width="180px"))
install_clang = W.Checkbox(value=True, description="Install Clang if missing")
install_libcxx = W.Checkbox(value=True, description="Install libc++ for Clang")
keep_workspace = W.Checkbox(value=True, description="Keep workspace")
continue_on_error = W.Checkbox(value=True, description="Continue after failures")

setup_rows=[]
test_rows=[]
setup_box=W.VBox()
test_box=W.VBox()

def make_setup_row(value=""):
    field=W.Text(value=value,placeholder="Optional setup shell command",layout=W.Layout(flex="1 1 320px",width="100%"))
    remove=W.Button(description="Remove",button_style="warning",layout=W.Layout(width="85px"))
    row=W.HBox([field,remove],layout=W.Layout(width="100%",flex_flow="row wrap"))
    item={"field":field,"row":row}
    remove.on_click(lambda _ : (setup_rows.remove(item), refresh_setup()))
    setup_rows.append(item)
    refresh_setup()
def refresh_setup():
    setup_box.children=tuple(item["row"] for item in setup_rows)

def make_test_row(name="",command="",needs=""):
    namef=W.Text(value=name,placeholder="Test name",layout=W.Layout(flex="1 1 140px"))
    cmdf=W.Textarea(value=command,placeholder="Command",layout=W.Layout(flex="3 1 260px",height="70px"))
    needsf=W.Text(value=needs,placeholder="Required executables (comma separated)",layout=W.Layout(flex="1 1 180px"))
    remove=W.Button(description="Remove",button_style="warning",layout=W.Layout(width="85px"))
    row=W.VBox([W.HBox([namef,needsf,remove],layout=W.Layout(width="100%",flex_flow="row wrap")),cmdf],layout=W.Layout(width="100%"))
    item={"name":namef,"command":cmdf,"needs":needsf,"row":row}
    remove.on_click(lambda _ : (test_rows.remove(item),refresh_tests()))
    test_rows.append(item)
    refresh_tests()
def refresh_tests():
    test_box.children=tuple(item["row"] for item in test_rows)

make_setup_row()
make_test_row("GCC · strict mock","bash verify/windows_mock/run_windows_mock_tests.sh","")
make_test_row("Clang · strict mock","EXTRA_CXXFLAGS='-stdlib=libc++' CXX=clang++ bash verify/windows_mock/run_windows_mock_tests.sh","clang++")
make_test_row("GCC · ASan + UBSan","EXTRA_CXXFLAGS='-fsanitize=address,undefined -fno-omit-frame-pointer' ASAN_OPTIONS='detect_leaks=1:halt_on_error=1' UBSAN_OPTIONS='halt_on_error=1:print_stacktrace=1' bash verify/windows_mock/run_windows_mock_tests.sh","")

add_setup=W.Button(description="+ Add setup command",button_style="info")
add_test=W.Button(description="+ Add test",button_style="info")
add_setup.on_click(lambda _ : make_setup_row())
add_test.on_click(lambda _ : make_test_row())

run_button=W.Button(description="▶ Run validation",button_style="success",icon="play",layout=W.Layout(width="100%",height="45px"))
output=W.Output(layout=W.Layout(width="100%"))
report_box=W.Textarea(value="",description="Full report",layout=W.Layout(width="100%",height="320px"))
report_box.add_class("ptc-report")

def redact(s, token):
    s=str(s or "")
    return s.replace(token,"[REDACTED]") if token else s
def clip(s,limit=18000):
    s=str(s or "")
    return s if len(s)<=limit else s[:limit//2]+"\n...[OUTPUT TRIMMED]...\n"+s[-limit//2:]
def shell(cmd,cwd,env,timeout):
    t=time.time()
    try:
        p=subprocess.run(cmd,cwd=str(cwd),env=env,shell=True,executable="/bin/bash",capture_output=True,text=True,timeout=timeout)
        out=(p.stdout or "")+("\n[STDERR]\n"+p.stderr if p.stderr else "")
        return p.returncode,round(time.time()-t,2),out
    except subprocess.TimeoutExpired as e:
        return 124,round(time.time()-t,2),f"TIMEOUT ({timeout}s)\n{e.stdout or ''}\n{e.stderr or ''}"
    except Exception as e:
        return -1,round(time.time()-t,2),f"{type(e).__name__}: {e}"

def run_validation(_):
    run_button.disabled=True
    run_button.description="Running…"
    with output:
        output.clear_output()
        display(HTML("<div style='padding:10px;border:1px solid #ccc;border-radius:10px'>Starting validation…</div>"))
    report_box.value=""
    start=time.time()
    issues=[]; setup_results=[]; results=[]
    work=pathlib.Path("/content/project-validation-workspace")
    token=None; clone_ok=False; commit="NOT_AVAILABLE"
    repository=repo.value.strip(); branch=ref.value.strip()
    try:
        if not repository.startswith(("https://","http://","git@")): raise ValueError("Repository URL must be https://, http:// or git@.")
        if depth.value<0 or clone_timeout.value<1 or test_timeout.value<1: raise ValueError("Depth and timeout values are invalid.")
        if not repository: raise ValueError("Repository is required.")
        if not test_rows: raise ValueError("Add at least one test.")
        tests=[]
        for i,item in enumerate(test_rows,1):
            n=item["name"].value.strip(); c=item["command"].value.strip()
            if n and c: tests.append({"name":n,"command":c,"needs":[x.strip() for x in item["needs"].value.split(",") if x.strip()]})
            elif n or c: issues.append({"stage":"Test configuration", "detail":f"Test row {i}: name and command are both required."})
        if not tests: raise ValueError("At least one complete test is required.")
    except Exception as e:
        issues.append({"stage":"Configuration","detail":str(e)})
        tests=[]
    if not issues:
        if work.exists(): shutil.rmtree(work,ignore_errors=True)
        env=os.environ.copy(); env["GIT_TERMINAL_PROMPT"]="0"; env["GCM_INTERACTIVE"]="Never"
        askdir=None
        try:
            if userdata is not None and secret_name.value.strip():
                try: token=userdata.get(secret_name.value.strip())
                except Exception: token=None
            cmd=["git","clone"]
            if depth.value>0: cmd += ["--depth",str(depth.value)]
            if branch: cmd += ["--single-branch","--branch",branch]
            cmd += [repository,str(work)]
            if token:
                askdir=pathlib.Path(tempfile.mkdtemp(prefix="ptc-askpass-"))
                ask=askdir/"askpass.sh"
                ask.write_text('#!/bin/sh\ncase "$1" in *Username*) echo "$PTC_USER";; *) echo "$PTC_TOKEN";; esac\n')
                ask.chmod(0o700)
                env.update({"GIT_ASKPASS":str(ask),"PTC_USER":username.value or "x-access-token","PTC_TOKEN":token})
            with output:
                output.clear_output()
                display(HTML("<div style='padding:10px;border:1px solid #ccc;border-radius:10px'>Cloning repository…</div>"))
            p=subprocess.run(cmd,env=env,capture_output=True,text=True,timeout=clone_timeout.value)
            if p.returncode:
                issues.append({"stage":"Clone","detail":redact(p.stderr or p.stdout,token)})
            else:
                clone_ok=True
                commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=work,text=True).strip()
        except subprocess.TimeoutExpired:
            issues.append({"stage":"Clone","detail":f"Clone timeout ({clone_timeout.value}s)"})
        except Exception as e:
            issues.append({"stage":"Clone","detail":f"{type(e).__name__}: {e}"})
        finally:
            if askdir: shutil.rmtree(askdir,ignore_errors=True)
            env.pop("PTC_TOKEN",None); env.pop("PTC_USER",None); token=None
    if clone_ok:
        env=os.environ.copy()
        for i,item in enumerate(setup_rows,1):
            cmd=item["field"].value.strip()
            if not cmd: continue
            code,duration,out=shell(cmd,work,env,test_timeout.value)
            record={"name":f"Setup {i}","command":cmd,"exit":code,"duration":duration,"output":clip(out)}
            setup_results.append(record)
            if code: issues.append({"stage":record["name"],"detail":record["output"] or f"exit={code}"})
            if code and not continue_on_error.value: break
        setup_blocked=any(x["exit"]!=0 for x in setup_results) and not continue_on_error.value
        clang_needed=any("clang++" in x["needs"] for x in tests)
        if not setup_blocked and install_clang.value and clang_needed and not shutil.which("clang++"):
            cmd="apt-get update -qq && apt-get install -y clang"
            c,d,o=shell(cmd,work,env,max(test_timeout.value,600))
            setup_results.append({"name":"Install Clang","command":cmd,"exit":c,"duration":d,"output":clip(o,3500)})
            if c: issues.append({"stage":"Install Clang","detail":clip(o,3500)})
        if not setup_blocked and install_libcxx.value and any("stdlib=libc++" in x["command"] for x in tests):
            if not pathlib.Path("/usr/include/c++/v1/expected").exists():
                cmd="apt-get update -qq && apt-get install -y libc++-18-dev libc++abi-18-dev"
                c,d,o=shell(cmd,work,env,max(test_timeout.value,600))
                setup_results.append({"name":"Install libc++","command":cmd,"exit":c,"duration":d,"output":clip(o,3500)})
                if c: issues.append({"stage":"Install libc++","detail":clip(o,3500)})
        for spec in tests:
            if setup_blocked:
                results.append({"name":spec["name"],"status":"SKIP","exit":None,"duration":0,"command":spec["command"],"output":"Skipped due to setup failure."}); continue
            missing=[n for n in spec["needs"] if shutil.which(n) is None]
            if missing:
                results.append({"name":spec["name"],"status":"SKIP","exit":None,"duration":0,"command":spec["command"],"output":"Missing executables: "+", ".join(missing)}); continue
            with output:
                output.clear_output()
                display(HTML(f"<div style='padding:10px;border:1px solid #ccc;border-radius:10px'>Running <b>{html.escape(spec['name'])}</b>…</div>"))
            c,d,o=shell(spec["command"],work,env,test_timeout.value)
            status="PASS" if c==0 else "FAIL"
            record={"name":spec["name"],"status":status,"exit":c,"duration":d,"command":spec["command"],"output":clip(redact(o,token))}
            results.append(record)
            if c:
                issues.append({"stage":spec["name"],"detail":record["output"] or f"exit={c}"})
                if not continue_on_error.value: break
    passed=sum(x["status"]=="PASS" for x in results); failed=sum(x["status"]=="FAIL" for x in results); skipped=sum(x["status"]=="SKIP" for x in results)
    setup_failed=any(x["exit"]!=0 for x in setup_results)
    overall="FAIL" if issues or failed or setup_failed or not clone_ok else ("PARTIAL" if skipped else ("PASS" if passed else "FAIL"))
    elapsed=round(time.time()-start,2)
    commit_url=""
    match=re.search(r"github\.com[:/]([^/]+/[^/.]+?)(?:\.git)?$",repository)
    if match and commit!="NOT_AVAILABLE": commit_url=f"https://github.com/{match.group(1)}/commit/{commit}"
    lines=["PROJECT VALIDATION REPORT",f"Overall: {overall}",f"Repository: {repository}",f"Branch: {branch or '(default branch)'}",f"Commit: {commit}",f"Commit URL: {commit_url or 'not available'}",f"Timestamp UTC: {datetime.datetime.now(datetime.timezone.utc).isoformat()}",f"Environment: {platform.platform()} | Python {platform.python_version()}",f"Duration: {elapsed}s",f"Results: {passed} PASS / {failed} FAIL / {skipped} SKIP","","SETUP RESULTS:"]
    if not setup_results: lines.append("(none)")
    for x in setup_results: lines += [f"[{'PASS' if x['exit']==0 else 'FAIL'}] {x['name']} | exit={x['exit']} | {x['duration']}s",f"COMMAND: {x['command']}",x["output"],""]
    lines.append("TEST RESULTS:")
    if not results: lines.append("(none)")
    for x in results: lines += [f"[{x['status']}] {x['name']} | exit={x['exit']} | {x['duration']}s",f"COMMAND: {x['command']}",x["output"],""]
    lines.append("ISSUE POOL:")
    if not issues: lines.append("(empty)")
    for i,x in enumerate(issues,1): lines += [f"--- ISSUE {i}: {x['stage']} ---",x["detail"],"--- END ISSUE ---",""]
    lines += ["","END OF REPORT"]
    report="\n".join(lines)
    pathlib.Path("/content/PROJECT_VALIDATION_REPORT.txt").write_text(report,encoding="utf-8")
    if clone_ok and not keep_workspace.value: shutil.rmtree(work,ignore_errors=True)
    color={"PASS":"#188038","FAIL":"#b3261e","PARTIAL":"#b06000"}.get(overall,"#666")
    esc=html.escape
    summary=f"""<div style="font-family:system-ui;max-width:100%;border:1px solid #ddd;border-radius:14px;padding:14px;box-sizing:border-box">
    <div style="font-size:22px;font-weight:750;color:{color}">{overall}</div><div style="display:flex;flex-wrap:wrap;gap:8px;margin:10px 0">
    <span style="padding:6px 10px;border-radius:20px;background:#e7f5e9">✓ {passed} PASS</span><span style="padding:6px 10px;border-radius:20px;background:#fde8e7">✕ {failed} FAIL</span><span style="padding:6px 10px;border-radius:20px;background:#eee">↷ {skipped} SKIP</span></div>
    <div style="overflow-wrap:anywhere;font-size:13px">Repository: {esc(repository)}<br>Ref: {esc(branch)}<br>Commit: {esc(commit)}<br>Elapsed: {elapsed}s</div>
    <h3>Test results</h3><div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:8px">"""
    cards=""
    for x in results:
        c={"PASS":"#188038","FAIL":"#b3261e","SKIP":"#666"}.get(x["status"],"#666")
        cards+=f"""<div style="border:1px solid #ddd;border-left:4px solid {c};border-radius:9px;padding:9px;overflow-wrap:anywhere"><b style="color:{c}">{x['status']}</b><br>{esc(x['name'])}<br><small>{x['duration']}s · exit {x['exit']}</small></div>"""
    fail_html=""
    for x in results:
        if x["status"]=="FAIL":
            fail_html+=f"<details open style='margin:8px 0'><summary><b>{esc(x['name'])}</b></summary><pre style='white-space:pre-wrap;overflow-wrap:anywhere;max-height:420px;overflow:auto;background:#f6f8fa;padding:10px;border-radius:8px'>{esc(x['output'])}</pre></details>"
    if not fail_html: fail_html="<p>No failing tests.</p>"
    report_b64=base64.b64encode(report.encode()).decode()
    summary+=cards+"</div><h3>Failure details</h3>"+fail_html+f"""
    <h3>Full report (tap and hold / Select all to copy)</h3><textarea id="ptc-report" readonly style="width:100%;height:280px;box-sizing:border-box;font:12px monospace;white-space:pre;overflow:auto;border:1px solid #ccc;border-radius:8px;padding:8px">{esc(report)}</textarea>
    <div style="display:flex;flex-wrap:wrap;gap:8px;margin-top:8px"><a download="PROJECT_VALIDATION_REPORT.txt" href="data:text/plain;charset=utf-8;base64,{report_b64}" style="display:inline-block;padding:10px 14px;background:#1967d2;color:white;border-radius:8px;text-decoration:none">Download TXT</a></div></div>"""
    with output:
        output.clear_output()
        display(HTML(summary))
    report_box.value=report
    run_button.disabled=False
    run_button.description="▶ Run validation"

run_button.on_click(run_validation)

header=W.HTML("<div style='font-family:system-ui'><h2 style='margin:0'>🧪 Project Test Console</h2><p>Responsive controls · Add or remove any number of setup commands and tests · Results designed for mobile copy/paste</p></div>")
project_card=W.VBox([W.HTML("<b>Project & access</b>"),repo,ref,secret_name,username,W.HBox([depth,clone_timeout,test_timeout],layout=W.Layout(flex_flow="row wrap")),W.HBox([install_clang,install_libcxx,keep_workspace,continue_on_error],layout=W.Layout(flex_flow="row wrap"))])
setup_card=W.VBox([W.HTML("<b>Setup commands</b><small>Optional. Add as many as needed. Runs in order before tests.</small>"),setup_box,add_setup])
test_card=W.VBox([W.HTML("<b>Test matrix</b><small>Each row has a name, shell command, and optional required executables.</small>"),test_box,add_test])
for card in [project_card,setup_card,test_card]: card.add_class("ptc-card")
display(style,header,project_card,setup_card,test_card,run_button,output)
